# Unsloth LLM Training Pipeline for Flight Rebooking

This notebook fine-tunes a Llama 3 model to act as the Flight Rebooking Agent using the synthetic expert trajectories generated by `generate_llm_dataset.py`.

It uses Unsloth for 2x faster training and less VRAM usage. It uses TRL's `SFTTrainer` for Supervised Fine-Tuning.

In [ ]:
%%capture
# Install Unsloth, Xformers, and TRL
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit", # Use the instruct model to inherit chat templates naturally
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from datasets import load_dataset
import json

# Load your generated JSONL dataset
# Ensure you upload artifacts/flight_rebooking_sft.jsonl to the Colab environment first
dataset = load_dataset("json", data_files="flight_rebooking_sft.jsonl", split="train")

def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        # We use the tokenizer's chat template directly
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched = True)
print("Sample training format:")
print(dataset[0]["text"])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Set up the Trainer
# Note: We log to Weights and Biases to satisfy the "Reward/Loss curve" hackathon requirement
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 150,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb", # High priority: Use WandB for the pitch demonstration
    ),
)

In [ ]:
import wandb
# You will be prompted to enter your WandB API key here so judges can see the graphs.
wandb.login()
wandb.init(project="flight-rebooking-agent", name="llama3-sft-run01")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# Saving the LoRA adapter
model.save_pretrained("flight-rebooking-lora") # Local saving
tokenizer.save_pretrained("flight-rebooking-lora")
# To test inference later locally or via HuggingFace Space, you will load this adapter.